# S6E11 | Own Models Only | Honest 5-Fold CV

*Playground Series, Season 6 Episode 11: Predicting Credit Card Fraud (ROC-AUC)*

This notebook trains **six models we wrote ourselves**, stacks them, and checks every step on the same
five folds. It uses no public submission files and no public out-of-fold predictions, and nothing is
chosen on the public leaderboard. It is not the top of the board, and it does not try to be: it is a
pipeline whose cross-validation you can trust, built from parts you can reuse.

**Why bother.** The public leaderboard scores 20% of the test rows, about 60,000 rows with about a
thousand frauds. Section 7 measures how much an AUC on that many rows moves by chance; at the top of
the board the steps are smaller than that. The private leaderboard (the other 80%) decides the ranking,
so here the five-fold CV decides every choice.

**What is in it.**

| Section | What it shows |
| --- | --- |
| [1 · Setup](#s1) | the library (the same code as the [GitHub repo](https://github.com/kabartay/playground-series-s6e11), MIT) |
| [2 · Data and metric](#s2) | class balance, the two scores that carry most of the signal |
| [3 · Folds, and the one rule](#s3) | everything learned from labels stays inside the training fold |
| [4 · Three GLMs](#s4) | target-encoded value keys, generator artefacts, GPT-2 token keys |
| [5 · Two shallow boosters](#s5) | 4-leaf LightGBM, and LightGBM started from the GLM ("GLM boost") |
| [6 · RealMLP](#s6) | a network on value-bucket embeddings |
| [7 · The stack](#s7) | nested logistic regression, and how noisy the public board is |
| [8 · Zero-fraud zones](#s8) | four regions with no fraud, checked with their edges refitted in each fold |
| [9 · What did not help](#s9) | measured in the repo, with numbers |
| [10 · Submission](#s10) | |

Run time on Kaggle CPU: about 2.5 hours, most of it the boosters and the RealMLP.

*Version 4 (2026-11-26):* the stacker and the zones now come from the repo's library (`stack.py`,
`zones.py`), section 7 adds a simulation of what public-board blending does to the private score, and
section 9 adds five more measured negatives. Scores are unchanged.

<a id="s1"></a>

## 1 · Setup

The models come from one training script, `train_gbdt.py`: feature builders in `features.py`, the
in-fold target encoder in `encoding.py`, one class per model family in `learners.py`, data loading,
folds and scores in `common.py`; the nested stacker comes from `stack.py` and the zero-fraud zones from
`zones.py`. They are written to disk below exactly as they are in the repo (where 65 unit tests cover
them), so every model in this notebook is one command line. `tiktoken` (MIT) gives GPT-2's tokeniser
for one feature family; `pytabkit` (Apache-2.0) provides RealMLP.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "pytabkit"], check=True)
import os
os.makedirs("src", exist_ok=True)
os.environ["S6E11_THREADS"] = "4"

In [ ]:
%%writefile src/common.py
"""Shared foundation for every script in ``src/``: paths, columns, data loading, folds, scores and runs.

Playground Series S6E11 (credit-card fraud, ROC-AUC). Everything that more than one script needs lives
here, so each idea is written once:

* **Paths.** ``DATA`` and ``RUNS`` follow ``S6E11_DATA`` / ``S6E11_RUNS`` and switch to the Kaggle layout
  when ``/kaggle/input`` exists; ``THREADS`` follows ``S6E11_THREADS`` (the Mac is shared: 4 or below).
* **Data.** :func:`load` returns train, test and the original dataset with one dtype convention.
* **Folds.** :func:`folds` is the one split every model and stack is scored on; never change its seed.
* **Scores.** :class:`CVScore` holds an out-of-fold AUC and its per-fold AUCs and compares two of them.
* **Rank transforms.** :func:`logit`, :func:`rank_fraction` and :func:`logit_rank`.
* **Runs.** :func:`save_run` and :func:`load_run` write and read ``runs/<name>/{oof,test}.npy`` and
  ``meta.json``.
"""

from __future__ import annotations

import json
import os
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.special import logit as special_logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path(__file__).resolve().parents[1]
ON_KAGGLE = Path("/kaggle/input").exists()
DATA = Path(os.environ.get("S6E11_DATA", ROOT / "data"))
RUNS = Path(os.environ.get("S6E11_RUNS", "/kaggle/working/runs" if ON_KAGGLE else ROOT / "runs"))
THREADS = int(os.environ.get("S6E11_THREADS", "4"))
"""Threads per model. The Mac is shared with a 24/7 pipeline: keep this at 4 or below."""

TARGET = "is_fraud"
N_FOLDS = 5
SEED = 42

CAT_STR = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
BOOL = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
NUM = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
RAW = NUM + CAT_STR + BOOL

Fold = tuple[np.ndarray, np.ndarray]
"""Row positions of one fold: (training rows, validation rows)."""


class ConfigError(ValueError):
    """A command-line option or input file that cannot work; scripts turn it into a one-line exit."""


# --------------------------------------------------------------------------------------------- data


def find_data(name: str, *alts: str) -> Path:
    """Locate a data file in ``DATA``, or anywhere under ``/kaggle/input`` on Kaggle.

    Args:
        name: preferred file name.
        *alts: other names the same file may have (the original dataset has two).

    Raises:
        FileNotFoundError: no candidate exists; the message names every place searched.
    """
    for n in (name, *alts):
        if (DATA / n).exists():
            return DATA / n
    if ON_KAGGLE:
        for n in (name, *alts):
            hits = sorted(Path("/kaggle/input").rglob(n))
            if hits:
                return hits[0]
    where = f"{DATA}" + (" or /kaggle/input" if ON_KAGGLE else "")
    raise FileNotFoundError(f"{' / '.join((name, *alts))} not found in {where} (see docs/DATA.md)")


def _bool_codes(s: pd.Series, column: str) -> pd.Series:
    """Map a True/False column (any case, or 1/0) to int8, refusing anything else."""
    out = s.astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).unique())[:5]
        raise ValueError(f"column {column}: values other than True/False: {bad}")
    return out.astype("int8")


def load() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (train, test, original) with booleans as int8 and string categoricals as pandas category.

    Category levels are the union over all three frames, so codes agree everywhere. The original
    dataset's ``transaction_id`` is replaced by negative ids, so ids never collide with competition rows.

    Raises:
        FileNotFoundError: a data file is missing.
        ValueError: a flag column holds something other than True/False.
    """
    tr = pd.read_csv(find_data("train.csv"))
    te = pd.read_csv(find_data("test.csv"))
    og = pd.read_csv(find_data("original.csv", "credit_card_fraud_2026.csv")).drop(columns=["transaction_id"])
    og.insert(0, "id", -1 - np.arange(len(og)))
    for df in (tr, te, og):
        for c in BOOL:
            df[c] = _bool_codes(df[c], c)
    for c in CAT_STR:
        levels = sorted(set(tr[c]) | set(te[c]) | set(og[c]))
        for df in (tr, te, og):
            df[c] = pd.Categorical(df[c], categories=levels)
    tr[TARGET] = tr[TARGET].astype(int)
    og[TARGET] = og[TARGET].astype(int)
    return tr, te, og


# -------------------------------------------------------------------------------------------- folds


def folds(y: np.ndarray, n: int = N_FOLDS, seed: int = SEED) -> list[Fold]:
    """The fixed stratified folds over the competition training rows (original rows never validate)."""
    return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def fold_ids(y: np.ndarray) -> np.ndarray:
    """Fold index (0-4) of every training row under :func:`folds`, as int8."""
    f = np.empty(len(y), dtype=np.int8)
    for k, (_, i_va) in enumerate(folds(y)):
        f[i_va] = k
    return f


# ------------------------------------------------------------------------------------------- scores


@dataclass(frozen=True)
class CVScore:
    """Out-of-fold ROC-AUC over all scored rows and the AUC of each validation fold."""

    cv: float
    folds: tuple[float, ...]

    @classmethod
    def of(
        cls, y: np.ndarray, pred: np.ndarray, fl: Sequence[Fold], rows: np.ndarray | None = None
    ) -> CVScore:
        """Score out-of-fold predictions.

        Args:
            y: labels of all training rows.
            pred: one out-of-fold prediction per training row.
            fl: the folds the predictions were made on.
            rows: optional boolean mask of scored rows (probe runs leave the others NaN).
        """
        if rows is None:
            return cls(
                float(roc_auc_score(y, pred)), tuple(float(roc_auc_score(y[v], pred[v])) for _, v in fl)
            )
        per_fold = tuple(float(roc_auc_score(y[v[rows[v]]], pred[v[rows[v]]])) for _, v in fl)
        return cls(float(roc_auc_score(y[rows], pred[rows])), per_fold)

    def up(self, base: CVScore) -> int:
        """How many folds score higher than in ``base`` (the adoption rule needs 4 of 5)."""
        return sum(a > b for a, b in zip(self.folds, base.folds, strict=True))

    def fold_list(self) -> list[float]:
        """Per-fold AUCs rounded to five decimals, for printing."""
        return np.round(self.folds, 5).tolist()

    def __str__(self) -> str:
        return f"{self.cv:.5f}  folds {self.fold_list()}"


# ---------------------------------------------------------------------------------- rank transforms


def logit(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Log-odds of probabilities clipped to ``[eps, 1 - eps]``."""
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def rank_fraction(v: np.ndarray) -> np.ndarray:
    """Mid-ranks scaled into (0, 1): ``(rank - 0.5) / n``; ties share their mean rank."""
    return (rankdata(v) - 0.5) / len(v)


def logit_rank(v: np.ndarray) -> np.ndarray:
    """Logit of :func:`rank_fraction`, so scores on any scale enter a blend on the same one."""
    return special_logit(rank_fraction(v))


# --------------------------------------------------------------------------------------------- runs


def save_run(name: str, oof: np.ndarray, test: np.ndarray, meta: dict[str, Any]) -> Path:
    """Write OOF and test predictions plus metadata to ``runs/<name>/`` and return that directory."""
    d = RUNS / name
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "oof.npy", oof)
    np.save(d / "test.npy", test)
    (d / "meta.json").write_text(json.dumps(meta, indent=2, default=str))
    return d


def load_run(name: str, refit: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """Read ``runs/<name>/{oof,test}.npy`` (never unpickles).

    Args:
        name: run name.
        refit: take the test scores from ``test_refit.npy`` (one model fitted on all rows, D34) when the run
            has one; runs without it (public members) keep ``test.npy``.

    Raises:
        FileNotFoundError: the run does not exist; the message names the run.
    """
    d = RUNS / name
    if not (d / "oof.npy").exists() or not (d / "test.npy").exists():
        raise FileNotFoundError(f"run {name!r} not found in {RUNS}")
    test = d / "test_refit.npy" if refit and (d / "test_refit.npy").exists() else d / "test.npy"
    return np.load(d / "oof.npy", allow_pickle=False), np.load(test, allow_pickle=False)

In [ ]:
%%writefile src/features.py
"""Feature sets and target-encoding keys, built once over train, test and the original dataset.

Two kinds of output:

* **Feature sets** (``FEATURE_SETS``, chosen with ``train_gbdt.py --fe``). Each builder takes
  ``(train, test, original)`` and returns three frames with identical columns.
* **Encoding keys** (:func:`te_keys`). Integer category codes, one column per key, that
  ``encoding.FoldEncoder`` target-encodes inside each fold.

Fold-safe by construction: nothing here reads the competition target. Target statistics come only from
the original dataset, which never appears in validation; everything else is unsupervised (value counts,
digits, tokens).
"""

from __future__ import annotations

import itertools
from collections.abc import Callable

import numpy as np
import pandas as pd

from common import BOOL, CAT_STR, NUM, RAW, TARGET

Frames = list[pd.DataFrame]
"""Three frames in the order (train, test, original)."""
FeatureBuilder = Callable[[pd.DataFrame, pd.DataFrame, pd.DataFrame], Frames]


def fe_raw(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """Raw columns only."""
    return [df[RAW].copy() for df in (tr, te, og)]


def fe_orig_te(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """Raw columns plus, for every column, the original dataset's fraud rate and count at that value.

    Unseen values get NaN. Smoothing towards the global rate with weight 20 rows.
    """
    out = fe_raw(tr, te, og)
    prior = og[TARGET].mean()
    for c in RAW:
        key = og[c].astype(str)
        g = og.groupby(key, observed=True)[TARGET].agg(["sum", "count"])
        rate = (g["sum"] + 20 * prior) / (g["count"] + 20)
        for src, dst in zip((tr, te, og), out, strict=True):
            k = src[c].astype(str)
            dst[f"ogte_{c}"] = k.map(rate).astype("float32").values
            dst[f"ogcnt_{c}"] = k.map(g["count"]).astype("float32").values
    return out


def fe_combo(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """fe_orig_te plus simple ratios and a count of risk flags."""
    out = fe_orig_te(tr, te, og)
    for src, dst in zip((tr, te, og), out, strict=True):
        dst["flags"] = src[BOOL].sum(axis=1).astype("int8")
        dst["amt_over_bal"] = (src.amount_usd / (src.account_balance_usd + 1)).astype("float32")
        dst["txn_per_hour"] = (src.txn_count_last_24h / (src.hours_since_last_txn + 0.1)).astype("float32")
        dst["risk_x_vel"] = (src.merchant_risk_score * src.velocity_score).astype("float32")
    return out


def fe_count(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """fe_combo plus frequency of each numeric value over train+test+original (unsupervised)."""
    out = fe_combo(tr, te, og)
    allv = pd.concat([tr[NUM], te[NUM], og[NUM]])
    for c in NUM:
        vc = allv[c].value_counts()
        for src, dst in zip((tr, te, og), out, strict=True):
            dst[f"freq_{c}"] = src[c].map(vc).astype("float32").values
    return out


def fe_ogmodel(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """fe_count plus the prediction of a LightGBM trained on the original dataset only.

    Train and test get a model fitted on all original rows; original rows get 5-fold out-of-fold
    predictions so none sees its own label. The competition target is never read.
    """
    import lightgbm as lgb
    from sklearn.model_selection import StratifiedKFold

    out = fe_count(tr, te, og)
    params = dict(
        objective="binary",
        learning_rate=0.03,
        n_estimators=400,
        num_leaves=15,
        min_child_samples=50,
        feature_fraction=0.7,
        verbose=-1,
        num_threads=4,
    )
    y = og[TARGET].to_numpy()
    oof = np.zeros(len(og))
    for i_tr, i_va in StratifiedKFold(5, shuffle=True, random_state=0).split(og, y):
        m = lgb.LGBMClassifier(**params).fit(og[RAW].iloc[i_tr], y[i_tr])
        oof[i_va] = m.predict_proba(og[RAW].iloc[i_va])[:, 1]
    m = lgb.LGBMClassifier(**params).fit(og[RAW], y)
    out[0]["ogmodel"] = m.predict_proba(tr[RAW])[:, 1].astype("float32")
    out[1]["ogmodel"] = m.predict_proba(te[RAW])[:, 1].astype("float32")
    out[2]["ogmodel"] = oof.astype("float32")
    return out


CONT = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
DIGIT_COLS = [
    "merchant_risk_score",
    "velocity_score",
    "hours_since_last_txn",
    "amount_usd",
    "distance_from_home_km",
    "account_balance_usd",
]
TE_TOP = [
    "cvv_retry_count",
    "ip_country_mismatch",
    "is_foreign_transaction",
    "is_new_merchant",
    "billing_shipping_mismatch",
    "auth_method",
    "used_vpn",
    "is_ai_generated_scam_attempt",
    "merchant_category",
    "txn_count_last_24h",
]


def digits(df: pd.DataFrame) -> pd.DataFrame:
    """First and second decimal digit of continuous columns: a modulo artefact of the synthesiser."""
    out: dict[str, np.ndarray] = {}
    for c in DIGIT_COLS:
        cents = np.rint(df[c].to_numpy() * 100).astype("int64")
        out[f"d1_{c}"] = ((cents // 10) % 10).astype("int8")
        out[f"d2_{c}"] = (cents % 10).astype("int8")
    return pd.DataFrame(out, index=df.index)


def fe_digits(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """Raw columns plus decimal digits."""
    return [
        pd.concat([r, digits(src)], axis=1) for r, src in zip(fe_raw(tr, te, og), (tr, te, og), strict=True)
    ]


JOINT = [
    ("category_risk", ("merchant_category", "merchant_risk_score")),
    ("foreign_ip", ("is_foreign_transaction", "ip_country_mismatch")),
    ("category_amount", ("merchant_category", "q:amount_usd")),
    ("txn_velocity", ("txn_count_last_24h", "q:velocity_score")),
    ("category_age", ("merchant_category", "customer_age")),
    ("amount_risk", ("q:amount_usd", "q:merchant_risk_score")),
    ("hours_velocity", ("q:hours_since_last_txn", "q:velocity_score")),
    ("age_risk", ("customer_age", "q:merchant_risk_score")),
]
"""Column pairs whose effect on fraud is not additive, as measured and published in megayak's notebook
"S6E11 | Per-Value + Digit TE | Shallow-Tree Stack" (Apache-2.0); "q:" means deciles of a float column."""


def token_frame(df: pd.DataFrame) -> dict[str, np.ndarray]:
    """Text keys of each float column's integer part: its first two GPT-2 tokens and its 3-digit chunks.

    The integer part is written as GPT-2 would see a number in running text (" 1371" for 1371.37), so
    the tokeniser's own digit grouping (" 13", "71") becomes a pair of keys; the 3-digit chunks ("137",
    "1") group the same digits a fixed way. Idea after aphoristicartist and busyaprime's public notebooks;
    written here independently. Tokens are computed once per distinct integer (tiktoken, MIT).
    """
    import tiktoken

    enc = tiktoken.get_encoding("gpt2")
    out: dict[str, np.ndarray] = {}
    for c in DIGIT_COLS:
        ints = pd.Series(np.floor(df[c].to_numpy()).astype("int64")).astype(str)
        uniq = pd.unique(ints)
        toks = {u: [enc.decode([t]) for t in enc.encode(" " + u)] for u in uniq}
        first = {u: t[0] for u, t in toks.items()}
        second = {u: (t[1] if len(t) > 1 else "") for u, t in toks.items()}
        out[f"k_{c}_g2a"] = ints.map(first).to_numpy()
        out[f"k_{c}_g2b"] = ints.map(second).to_numpy()
        out[f"k_{c}_c3a"] = ints.str[:3].to_numpy()
        out[f"k_{c}_c3b"] = ints.str[3:].to_numpy()
    return out


def te_keys(
    tr: pd.DataFrame,
    te: pd.DataFrame,
    og: pd.DataFrame,
    pairs: bool,
    digit_keys: bool = False,
    joint: bool = False,
    token_keys: bool = False,
) -> Frames:
    """Integer category keys for in-fold target encoding (train, test, original), built on the union of rows.

    Every column's exact value is a key (when it has at most half as many levels as rows); continuous
    columns are also keyed on rounded values; decimal endings are keys; with `pairs`, every pair among
    TE_TOP is a key; with `joint`, every pair in JOINT is a key named j_<name>; with `token_keys`, the first
    two GPT-2 tokens and the 3-digit chunks of each float column's integer part are keys (`token_frame`).
    The encoding itself happens inside each fold (``encoding.FoldEncoder``).
    """
    both = pd.concat([tr[RAW], te[RAW], og[RAW]], ignore_index=True)
    keys: dict[str, np.ndarray] = {}
    for c in RAW:
        codes, uniq = pd.factorize(both[c].astype(str))
        if len(uniq) <= len(both) // 2:
            keys[f"k_{c}"] = codes
    for c in CONT:
        keys[f"k_{c}_int"] = pd.factorize(np.round(both[c]).astype("int64"))[0]
        keys[f"k_{c}_r10"] = pd.factorize(np.round(both[c] / 10).astype("int64"))[0]
    for c in ["merchant_risk_score", "velocity_score"]:
        keys[f"k_{c}_int"] = pd.factorize(np.round(both[c]).astype("int64"))[0]
    for c in DIGIT_COLS:
        keys[f"k_{c}_end"] = np.rint(both[c].to_numpy() * 100).astype("int64") % 100
    if digit_keys:
        # The digit-key family of the six float columns: integer part, tens, value at 0.1 resolution,
        # first one and two significant digits, and the length of the printed value.
        for c in DIGIT_COLS:
            x = both[c].to_numpy()
            text = both[c].astype(str)
            sig = text.str.replace(".", "", regex=False).str.lstrip("0-")
            keys[f"k_{c}_floor"] = pd.factorize(np.floor(x).astype("int64"))[0]
            keys[f"k_{c}_ten"] = pd.factorize(np.floor(x / 10).astype("int64"))[0]
            keys[f"k_{c}_dec"] = pd.factorize(np.floor(x * 10 + 1e-6).astype("int64"))[0]
            keys[f"k_{c}_lead1"] = pd.factorize(sig.str[:1])[0]
            keys[f"k_{c}_lead2"] = pd.factorize(sig.str[:2])[0]
            keys[f"k_{c}_len"] = text.str.len().to_numpy()
    if token_keys:
        for name, col in token_frame(both).items():
            keys[name] = pd.factorize(col)[0]
    if pairs:
        for a, b in itertools.combinations(TE_TOP, 2):
            keys[f"k_{a}__{b}"] = pd.factorize(both[a].astype(str) + "_" + both[b].astype(str))[0]
    if joint:
        for name, cols in JOINT:
            parts = [
                pd.qcut(both[c[2:]], 10, labels=False, duplicates="drop").astype(str)
                if c.startswith("q:")
                else both[c].astype(str)
                for c in cols
            ]
            keys[f"j_{name}"] = pd.factorize(parts[0] + "_" + parts[1])[0]
    k = pd.DataFrame(keys).astype("int32")
    n1, n2 = len(tr), len(tr) + len(te)
    return [
        k.iloc[:n1].reset_index(drop=True),
        k.iloc[n1:n2].reset_index(drop=True),
        k.iloc[n2:].reset_index(drop=True),
    ]


ART_PAIR_COLS = [
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
    "velocity_score",
    "merchant_risk_score",
]


def fe_art(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """Raw columns plus pooled generator artefacts (unsupervised, no labels used).

    For each numeric column with more than 100 distinct values: log1p of how often the exact value occurs
    in train + test (`art_cnt_*`) and whether it occurs in the original data (`art_inog_*`). For each pair
    of ART_PAIR_COLS: whether the pair of values occurs together in one original row (`art_pair_*`).
    Measured in `research_artefacts.py` (D14): such pooled effects are invisible to per-value encodings.
    """
    out = fe_raw(tr, te, og)
    allv = pd.concat([tr[NUM], te[NUM]])
    for c in NUM:
        if allv[c].nunique() <= 100:
            continue
        vc = allv[c].value_counts()
        ogv = set(og[c].unique())
        for src, dst in zip((tr, te, og), out, strict=True):
            dst[f"art_cnt_{c}"] = np.log1p(src[c].map(vc).fillna(0).to_numpy()).astype("float32")
            dst[f"art_inog_{c}"] = src[c].isin(ogv).to_numpy().astype("int8")
    for a, b in itertools.combinations(ART_PAIR_COLS, 2):
        ok = set(zip(og[a], og[b], strict=True))
        for src, dst in zip((tr, te, og), out, strict=True):
            hit = np.fromiter((p in ok for p in zip(src[a], src[b], strict=True)), bool, len(src))
            dst[f"art_pair_{a}__{b}"] = hit.astype("int8")
    return out


def fe_emb(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> Frames:
    """Raw columns plus categorical copies of every numeric column, for networks that embed categories.

    `emb_<col>`: the integer part of each numeric column as a category (levels pooled over train + test +
    original); for each DIGIT_COLS column also `emb_dec_<col>`, its two decimals as a category, and
    `dec_<col>`, the decimals as a number. Unsupervised. Idea from yekenot's public RealMLP notebook (read as
    text): the network learns an embedding per value bucket instead of seeing target encodings.
    """
    out = fe_raw(tr, te, og)
    srcs = (tr, te, og)
    for c in NUM:
        keys = [np.floor(s[c].to_numpy()).astype("int64") for s in srcs]
        levels = np.unique(np.concatenate(keys))
        for k, dst in zip(keys, out, strict=True):
            dst[f"emb_{c}"] = pd.Categorical(k, categories=levels)
    for c in DIGIT_COLS:
        keys = [np.rint((s[c].to_numpy() % 1) * 100).astype("int64") for s in srcs]
        levels = np.unique(np.concatenate(keys))
        for k, dst in zip(keys, out, strict=True):
            dst[f"emb_dec_{c}"] = pd.Categorical(k, categories=levels)
            dst[f"dec_{c}"] = (k / 100).astype("float32")
    return out


FEATURE_SETS: dict[str, FeatureBuilder] = {
    "raw": fe_raw,
    "art": fe_art,
    "ogte": fe_orig_te,
    "combo": fe_combo,
    "count": fe_count,
    "ogmodel": fe_ogmodel,
    "digits": fe_digits,
    "emb": fe_emb,
}
CATS = CAT_STR
"""String categoricals as the models see them (pandas category columns)."""

In [ ]:
%%writefile src/encoding.py
"""In-fold target encoding of the keys from ``features.te_keys``, the core of every strong model here.

The data is synthetic, and its generator reuses the original dataset's values: the exact value of a column,
its cents, its integer part and its leading digits all carry fraud signal that trees cannot express with
range splits. Each such key is target-encoded **inside the fold**:

* the encoder is fitted on the fold's competition training rows only;
* the training rows' own encodings are cross-fitted (``cv`` inner folds), so no row sees its own label;
* validation, test and original rows are encoded by the encoder fitted on all of the fold's training rows.

:class:`FoldEncoder` does this for one or several m-estimate smoothing weights and, optionally, adds
hierarchical encodings of each float column's coarse-to-fine key chain (:func:`hier_te`). :func:`key_counts`
adds the unsupervised log count of every key's value.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np
import pandas as pd

from common import ConfigError
from features import DIGIT_COLS, Frames

HIER_LEVELS = ("ten", "floor", "dec", "")
"""Coarse-to-fine keys of one float column for the hierarchical encoding: tens, integer part, 0.1 steps,
exact value ("" is the exact-value key k_<col>). Each level nests inside the one before it."""


def chain_rates(Kfit: np.ndarray, yfit: np.ndarray, Kapp: np.ndarray, m: float) -> np.ndarray:
    """Fraud rate of each applied row at every level of a key chain, shrunk towards the parent level.

    Args:
        Kfit: integer codes of the fit rows (rows x levels, coarse to fine).
        yfit: labels of the fit rows.
        Kapp: integer codes of the rows to encode, same levels.
        m: m-estimate weight. Level l's rate is (frauds + m * parent rate) / (rows + m) over the fit rows
            sharing the key, starting from the overall rate.

    Returns:
        Rates, applied rows x levels.
    """
    r = np.full(len(Kapp), yfit.mean())
    out = np.empty((len(Kapp), Kfit.shape[1]))
    for lvl in range(Kfit.shape[1]):
        size = int(max(Kfit[:, lvl].max(), Kapp[:, lvl].max())) + 1
        s = np.bincount(Kfit[:, lvl], weights=yfit, minlength=size)
        n = np.bincount(Kfit[:, lvl], minlength=size)
        c = Kapp[:, lvl]
        r = (s[c] + m * r) / (n[c] + m)
        out[:, lvl] = r
    return out


def hier_te(
    Ktr: pd.DataFrame, ytr: np.ndarray, others: Sequence[pd.DataFrame], m: float, cv: int, seed: int = 0
) -> Frames:
    """Hierarchical target encoding of the DIGIT_COLS key chains (HIER_LEVELS), fitted on the training rows.

    Training rows are cross-fitted over ``cv`` inner folds; the ``others`` frames use all training rows.
    Columns are named te_h_<col>_<level>. Idea after busyaprime's public "hier" members; written here
    independently.

    Raises:
        ConfigError: no column has a key chain (the digit-key family is missing: use ``--keys full``).
    """
    from sklearn.model_selection import StratifiedKFold

    blocks: list[list[np.ndarray]] = [[] for _ in range(1 + len(others))]
    names: list[str] = []
    for col in DIGIT_COLS:
        keys = [f"k_{col}_{lv}" if lv else f"k_{col}" for lv in HIER_LEVELS]
        keys = [k for k in keys if k in Ktr.columns]
        if len(keys) < 2:
            continue
        names += [f"te_h_{k[2:]}" for k in keys]
        A = Ktr[keys].to_numpy()
        inner = np.empty((len(A), len(keys)))
        for i_fit, i_app in StratifiedKFold(cv, shuffle=True, random_state=seed).split(A, ytr):
            inner[i_app] = chain_rates(A[i_fit], ytr[i_fit], A[i_app], m)
        blocks[0].append(inner)
        for j, K in enumerate(others, start=1):
            blocks[j].append(chain_rates(A, ytr, K[keys].to_numpy(), m))
    if not names:
        raise ConfigError("hierarchical encoding needs the digit-key family (--keys full)")
    return [pd.DataFrame(np.hstack(b).astype("float32"), columns=names) for b in blocks]


def key_counts(features: Frames, keys: Frames) -> Frames:
    """Append ``art_kc_<key>``: log1p of how many train + test rows share each key's value. Unsupervised.

    Args:
        features: (train, test, original) feature frames.
        keys: the matching (train, test, original) key frames.

    Returns:
        New (train, test, original) frames with one count column per key, in key order.
    """
    pooled = pd.concat([keys[0], keys[1]], ignore_index=True)
    counts = {c: pooled[c].value_counts() for c in keys[0].columns}
    return [
        pd.concat(
            [
                X,
                pd.DataFrame(
                    {
                        f"art_kc_{c}": np.log1p(K[c].map(vc).fillna(0).to_numpy()).astype("float32")
                        for c, vc in counts.items()
                    },
                    index=X.index,
                ),
            ],
            axis=1,
        )
        for X, K in zip(features, keys, strict=True)
    ]


class FoldEncoder:
    """Target-encode the key frames inside one fold and join the encodings to the feature frames.

    Args:
        keys: (train, test, original) key frames from ``features.te_keys``.
        y: labels of the training rows.
        smooths: m-estimate weights as given on the command line ("auto" or numbers). The first gives
            columns te_<key>, the others te_<key>_m<weight>.
        cv: inner cross-fitting folds for the training rows' own encodings.
        seed: seed of the inner split.
        glm_only_joint: name joint keys (j_*) tej_<key>, so that only GLM-design models read them.
        hier: m-estimate weight of the hierarchical encodings; 0 adds none.
    """

    def __init__(
        self,
        keys: Frames,
        y: np.ndarray,
        smooths: Sequence[str],
        cv: int = 5,
        seed: int = 0,
        glm_only_joint: bool = False,
        hier: float = 0.0,
    ) -> None:
        if len(keys) != 3:
            raise ConfigError("FoldEncoder needs (train, test, original) key frames")
        if not smooths:
            raise ConfigError("at least one smoothing weight is needed (--te-smooth)")
        self.Ktr, self.Kte, self.Kog = keys
        self.y = y
        self.smooths = list(smooths)
        self.cv = cv
        self.seed = seed
        self.glm_only_joint = glm_only_joint
        self.hier = hier

    @property
    def n_keys(self) -> int:
        """Number of key columns encoded per smoothing weight."""
        return self.Ktr.shape[1]

    def _names(self, suffix: str) -> list[str]:
        pre = ["tej_" if self.glm_only_joint and c.startswith("j_") else "te_" for c in self.Ktr.columns]
        return [f"{p}{c}{suffix}" for p, c in zip(pre, self.Ktr.columns, strict=True)]

    def encodings(self, i_tr: np.ndarray, i_va: np.ndarray) -> list[Frames]:
        """Encoding frames for (training, validation, test, original) rows of one fold."""
        from sklearn.preprocessing import TargetEncoder

        frames: list[Frames] = [[], [], [], []]
        for j, sm in enumerate(self.smooths):
            smooth = sm if sm == "auto" else float(sm)
            enc = TargetEncoder(target_type="binary", smooth=smooth, cv=self.cv, random_state=self.seed)
            cols = self._names("" if j == 0 else f"_m{sm}")
            A = enc.fit_transform(self.Ktr.iloc[i_tr], self.y[i_tr]).astype("float32")
            for k, K in enumerate((None, self.Ktr.iloc[i_va], self.Kte, self.Kog)):
                E = A if K is None else enc.transform(K).astype("float32")
                frames[k].append(pd.DataFrame(E, columns=cols))
        if self.hier > 0:
            others = [self.Ktr.iloc[i_va], self.Kte, self.Kog]
            H = hier_te(self.Ktr.iloc[i_tr], self.y[i_tr], others, self.hier, self.cv, self.seed)
            for k, h in enumerate(H):
                frames[k].append(h)
        return frames

    def transform(
        self, features: Frames, i_tr: np.ndarray, i_va: np.ndarray
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Feature frames of one fold with its encodings appended: (training, validation, test, original).

        Args:
            features: (train, test, original) feature frames.
            i_tr, i_va: the fold's training and validation row positions.
        """
        Xtr, Xte, Xog = features
        bases = (Xtr.iloc[i_tr], Xtr.iloc[i_va], Xte, Xog)
        parts = [
            pd.concat([X.reset_index(drop=True), *F], axis=1)
            for X, F in zip(bases, self.encodings(i_tr, i_va), strict=True)
        ]
        return parts[0], parts[1], parts[2], parts[3]

In [ ]:
%%writefile src/learners.py
"""Model families behind ``train_gbdt.py --model``: one class per family, one interface for all.

Every learner trains on one fold and predicts that fold's validation rows and the test set:

    learner = make_learner("lgb", {"num_leaves": 4})
    pred = learner.fit_predict(fold_data, seed=0)          # Prediction(val, test, n_iter)

Families (``LEARNERS``):

==========  ======================================================  =========================
key         model                                                   output scale
==========  ======================================================  =========================
``glm``     L2 logistic regression on :func:`glm_design`            logit
``mlp``     scikit-learn MLP on the same design                     probability
``realmlp`` pytabkit RealMLP (or another pytabkit classifier)       probability
``lgb``     LightGBM                                                probability, logit if GLM boost
``xgb``     XGBoost                                                 probability, logit if GLM boost
``cat``     CatBoost                                                probability, logit if GLM boost
==========  ======================================================  =========================

**GLM boost.** When :attr:`FoldData.init` holds a (train, validation, test) tuple of logits, the boosters
start from it (LightGBM ``init_score``, XGBoost ``base_margin``, CatBoost ``baseline``) and predict that
logit plus the trees' raw score.

**Early stopping.** Boosters early-stop on the validation fold unless ``n_iter`` is given, in which case
they train exactly that many rounds and never see its labels. :meth:`Learner.inner_rounds` chooses
``n_iter`` on a holdout inside the training fold (``--es inner``, honest out-of-fold scores).

Stack consumers tell the two output scales apart by range (``stack.to_logit``).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from collections.abc import Callable
from dataclasses import dataclass, replace
from typing import Any, ClassVar, NamedTuple

import numpy as np
import pandas as pd

from common import BOOL, NUM, THREADS, ConfigError
from features import CATS

Offsets = tuple[np.ndarray, np.ndarray, np.ndarray]
"""Starting logits of GLM boost for (training, validation, test) rows."""
Params = dict[str, Any]


@dataclass(frozen=True)
class FoldData:
    """Everything one fold's model sees.

    Attributes:
        X, y, w: training rows, labels and sample weights.
        X_val, y_val: validation rows and labels; ``y_val`` is None when the labels must stay unseen
            (``--es inner``).
        X_test: test rows.
        init: optional GLM-boost starting logits.
    """

    X: pd.DataFrame
    y: np.ndarray
    w: np.ndarray
    X_val: pd.DataFrame
    y_val: np.ndarray | None
    X_test: pd.DataFrame
    init: Offsets | None = None

    def training_subset(self, rows: np.ndarray) -> FoldData:
        """The same fold restricted to some training rows (validation and test unchanged)."""
        init = None if self.init is None else (self.init[0][rows], self.init[1], self.init[2])
        return replace(self, X=self.X.iloc[rows], y=self.y[rows], w=self.w[rows], init=init)


class Prediction(NamedTuple):
    """Validation and test predictions of one fold, and the boosting rounds (or solver iterations) used."""

    val: np.ndarray
    test: np.ndarray
    n_iter: int


# ------------------------------------------------------------------------------------- GLM design


def glm_design(X: pd.DataFrame) -> np.ndarray:
    """Logit of every target-encoded column, one-hot categoricals, the 0/1 flags and any art_ features."""
    te_cols = [c for c in X.columns if c.startswith(("te_", "tej_"))]
    q = np.clip(X[te_cols].to_numpy(dtype="float64"), 1e-5, 1 - 1e-5)
    parts = [
        np.log(q / (1 - q)),
        pd.get_dummies(X[[c for c in CATS if c in X.columns]], dtype="float32").to_numpy(),
    ]
    flags = [c for c in BOOL if c in X.columns]
    parts.append(X[flags].to_numpy(dtype="float32"))
    art = [c for c in X.columns if c.startswith("art_")]
    if art:
        parts.append(X[art].to_numpy(dtype="float32"))
    return np.hstack(parts)


def make_design(params: Params, Xtr: pd.DataFrame) -> Callable[[pd.DataFrame], np.ndarray]:
    """Return a function mapping a frame to the GLM design: :func:`glm_design` plus optional splines.

    Args:
        params: ``splines`` is the number of knots per numeric column (0, the default, adds none).
        Xtr: training frame; the quantile transform and the splines are fitted on it only.
    """
    from sklearn.preprocessing import QuantileTransformer, SplineTransformer

    knots = int(params.get("splines", 0))
    nums = [c for c in NUM if c in Xtr.columns]
    if knots <= 0 or not nums:
        return glm_design
    qt = QuantileTransformer(n_quantiles=1000, subsample=200_000, random_state=0)
    st = SplineTransformer(n_knots=knots, degree=3)
    st.fit(qt.fit_transform(Xtr[nums].to_numpy(dtype="float64")))

    def design(X: pd.DataFrame) -> np.ndarray:
        return np.hstack([glm_design(X), st.transform(qt.transform(X[nums].to_numpy(dtype="float64")))])

    return design


def fit_glm(
    params: Params, Xtr: pd.DataFrame, ytr: np.ndarray, Xs: list[pd.DataFrame]
) -> tuple[list[np.ndarray], int]:
    """L2 logistic regression on the GLM design; return logits for each frame in ``Xs`` and the iterations.

    Also the starting model of GLM boost (``train_gbdt.py --offset glm``).

    Args:
        params: ``C`` (default 0.01), ``splines`` (see :func:`make_design`), ``solver`` and ``tol``
            (default lbfgs and 1e-4, scikit-learn's defaults; D29 checked lbfgs convergence).
        Xtr, ytr: training rows and labels.
        Xs: frames to score.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    design = make_design(params, Xtr)
    Dtr = design(Xtr)
    sc = StandardScaler().fit(Dtr)
    solver, tol = params.get("solver", "lbfgs"), params.get("tol", 1e-4)
    m = LogisticRegression(C=params.get("C", 0.01), solver=solver, tol=tol, max_iter=3000)
    m.fit(sc.transform(Dtr), ytr)
    return [m.decision_function(sc.transform(design(X))) for X in Xs], int(m.n_iter_[0])


# ---------------------------------------------------------------------------------------- learners


class Learner(ABC):
    """One model family. Subclasses set :attr:`key` and implement :meth:`fit_predict`.

    Args:
        params: model options from ``--params``, overriding the family's defaults.
    """

    key: ClassVar[str]
    """Name on the command line (``--model``)."""
    booster: ClassVar[bool] = False
    """Accepts ``n_iter`` and GLM-boost offsets; ``--es inner`` applies."""
    seeded: ClassVar[bool] = True
    """Results depend on the seed, so ``--seeds K`` averages K models."""
    on_glm_design: ClassVar[bool] = False
    """Reads the GLM design, so it keeps the GLM-only joint encodings (``tej_`` columns)."""
    best_iter_base: ClassVar[int] = 1
    """1 if the reported best iteration is a round count, 0 if it is a 0-based index (CatBoost)."""

    def __init__(self, params: Params | None = None) -> None:
        self.params: Params = dict(params or {})

    @abstractmethod
    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """Train on ``d``'s training rows and predict its validation and test rows.

        Args:
            d: the fold.
            seed: the model's random seed.
            n_iter: boosters only: train exactly this many rounds and never read ``d.y_val``.
        """

    def inner_rounds(self, d: FoldData, seed: int, fold: int) -> int:
        """Boosting rounds for one fold, chosen without the validation fold.

        Early-stops on a stratified 10% holdout of the training rows, then scales the best round count by
        1/0.9 for the refit on all training rows. With GLM boost the holdout's starting logits come from
        the fold's GLM, which saw those rows; that can only shift the round count, never the validation fold.

        Raises:
            ConfigError: the family has no boosting rounds.
        """
        from sklearn.model_selection import train_test_split

        if not self.booster:
            raise ConfigError(f"--es inner needs a booster, not {self.key}")
        inn, hold = train_test_split(
            np.arange(len(d.y)), test_size=0.1, stratify=d.y, random_state=100 + fold
        )
        init = None if d.init is None else (d.init[0][inn], d.init[0][hold], d.init[0][hold])
        X_hold = d.X.iloc[hold]
        holdout = FoldData(d.X.iloc[inn], d.y[inn], d.w[inn], X_hold, d.y[hold], X_hold, init)
        best = int(self.fit_predict(holdout, seed=seed).n_iter) + (1 - self.best_iter_base)
        return max(1, round(best / 0.9))


class GLMLearner(Learner):
    """L2 logistic regression on the GLM design (:func:`fit_glm`); predicts logits."""

    key = "glm"
    seeded = False
    on_glm_design = True

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`; ``seed`` and ``n_iter`` do not apply."""
        (lv, lt), it = fit_glm(self.params, d.X, d.y, [d.X_val, d.X_test])
        return Prediction(lv, lt, it)


class MLPLearner(Learner):
    """Small ReLU network (scikit-learn) on the standardised GLM design; predicts probabilities.

    params: ``splines`` (see :func:`make_design`), ``hidden`` (layer sizes, default [64, 32]), ``alpha``
    (L2, default 1e-3), ``lr`` (default 1e-3), ``max_iter`` (default 40). Early stopping on an internal 10%
    split of the training rows.
    """

    key = "mlp"
    on_glm_design = True

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`; ``n_iter`` does not apply."""
        from sklearn.neural_network import MLPClassifier
        from sklearn.preprocessing import StandardScaler

        p = self.params
        design = make_design(p, d.X)
        Dtr = design(d.X)
        sc = StandardScaler().fit(Dtr)
        m = MLPClassifier(
            hidden_layer_sizes=tuple(p.get("hidden", [64, 32])),
            alpha=p.get("alpha", 1e-3),
            learning_rate_init=p.get("lr", 1e-3),
            batch_size=2048,
            max_iter=p.get("max_iter", 40),
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5,
            random_state=seed,
        )
        m.fit(sc.transform(Dtr), d.y)
        val, test = (m.predict_proba(sc.transform(design(X)))[:, 1] for X in (d.X_val, d.X_test))
        return Prediction(val, test, int(m.n_iter_))


class RealMLPLearner(Learner):
    """pytabkit's RealMLP_TD_Classifier on the fold's feature frame; predicts probabilities.

    pytabkit picks its best epoch (and early-stops, if asked) on a validation set. That set is a stratified
    10% holdout of the training rows, never the validation fold, so the OOF stays honest; the network trains
    on the other 90%. ``params`` go to the classifier unchanged (n_ens, n_epochs, batch_size, lr,
    hidden_sizes, ...), except:

    * ``cls``: another pytabkit classifier with the same interface (e.g. ``TabM_D_Classifier``);
    * ``holdout`` (default 0.1): the holdout share. 0 trains on all training rows and hands pytabkit a 5%
      stratified subset of those same rows as its validation set, so any epoch choice is made on training
      rows (still never the validation fold); meant for fixed-epoch runs without early stopping.

    Categoricals are the pandas category columns. Needs ``pip install pytabkit``; sample weights are not
    supported (``--orig 0``).
    """

    key = "realmlp"

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`; ``n_iter`` does not apply (the epoch count is a param)."""
        import pytabkit
        import torch
        from sklearn.model_selection import train_test_split

        torch.set_num_threads(THREADS)
        params = dict(self.params)
        share = float(params.pop("holdout", 0.1))
        idx = np.arange(len(d.y))
        if share > 0:
            inn, hold = train_test_split(idx, test_size=share, stratify=d.y, random_state=100 + seed)
        else:
            inn = idx
            hold = train_test_split(idx, test_size=0.05, stratify=d.y, random_state=100 + seed)[1]
        Xtr = d.X.reset_index(drop=True)
        cls = getattr(pytabkit, params.pop("cls", "RealMLP_TD_Classifier"))
        p = dict(device="cpu", n_threads=THREADS, random_state=seed, verbosity=0)
        p.update(params)
        m = cls(**p)
        m.fit(Xtr.iloc[inn], d.y[inn], Xtr.iloc[hold], d.y[hold])
        return Prediction(
            m.predict_proba(d.X_val)[:, 1], m.predict_proba(d.X_test)[:, 1], int(p.get("n_epochs", 0))
        )


class LightGBMLearner(Learner):
    """LightGBM (``LGBMClassifier``); with GLM-boost offsets it predicts logits."""

    key = "lgb"
    booster = True
    DEFAULTS: ClassVar[Params] = dict(
        objective="binary",
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=100,
        feature_fraction=0.5,
        bagging_fraction=0.8,
        bagging_freq=1,
        lambda_l2=5.0,
        cat_smooth=20,
        n_estimators=5000,
    )

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`; early-stops after 200 rounds without AUC gain."""
        import lightgbm as lgb

        p = dict(self.DEFAULTS, num_threads=THREADS, verbose=-1, random_state=seed)
        p.update(self.params)
        if n_iter is not None:
            p["n_estimators"] = n_iter
        m = lgb.LGBMClassifier(**p)
        if n_iter is None:
            extra = {} if d.init is None else {"init_score": d.init[0], "eval_init_score": [d.init[1]]}
            m.fit(
                d.X,
                d.y,
                sample_weight=d.w,
                eval_set=[(d.X_val, d.y_val)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(200, verbose=False)],
                **extra,
            )
            best = m.best_iteration_
        else:
            m.fit(d.X, d.y, sample_weight=d.w, **({} if d.init is None else {"init_score": d.init[0]}))
            best = n_iter
        if d.init is not None:
            return Prediction(
                m.predict(d.X_val, raw_score=True) + d.init[1],
                m.predict(d.X_test, raw_score=True) + d.init[2],
                best,
            )
        return Prediction(m.predict_proba(d.X_val)[:, 1], m.predict_proba(d.X_test)[:, 1], best)


class XGBoostLearner(Learner):
    """XGBoost (``XGBClassifier``, hist, native categoricals); with GLM-boost offsets it predicts logits.

    A ``rank:*`` objective (D30) takes one weight per query group, so it needs unit sample weights.
    """

    key = "xgb"
    booster = True
    DEFAULTS: ClassVar[Params] = dict(
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_lambda=5.0,
        n_estimators=5000,
        tree_method="hist",
        enable_categorical=True,
        max_cat_to_onehot=1,
        eval_metric="auc",
        early_stopping_rounds=200,
    )

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`.

        Raises:
            ConfigError: a ranking objective with non-unit sample weights (``--orig`` above 0).
        """
        import xgboost as xgb

        p = dict(self.DEFAULTS, n_jobs=THREADS, random_state=seed)
        p.update(self.params)
        if n_iter is not None:
            p.update(n_estimators=n_iter, early_stopping_rounds=None)
        m = xgb.XGBClassifier(**p)
        w: np.ndarray | None = d.w
        if str(p.get("objective", "")).startswith("rank:"):
            # Ranking objectives take one weight per query group (here the whole fold is one group).
            if w is not None and not np.all(w == 1):
                raise ConfigError("ranking objectives need unit weights (--orig 0)")
            w = None
        if n_iter is None:
            extra = {} if d.init is None else {"base_margin": d.init[0], "base_margin_eval_set": [d.init[1]]}
            m.fit(d.X, d.y, sample_weight=w, eval_set=[(d.X_val, d.y_val)], verbose=False, **extra)
            it = m.best_iteration + 1
        else:
            extra = {} if d.init is None else {"base_margin": d.init[0]}
            m.fit(d.X, d.y, sample_weight=w, verbose=False, **extra)
            it = n_iter
        if d.init is not None:
            return Prediction(
                m.predict(d.X_val, output_margin=True, base_margin=d.init[1], iteration_range=(0, it)),
                m.predict(d.X_test, output_margin=True, base_margin=d.init[2], iteration_range=(0, it)),
                it,
            )
        return Prediction(
            m.predict_proba(d.X_val, iteration_range=(0, it))[:, 1],
            m.predict_proba(d.X_test, iteration_range=(0, it))[:, 1],
            it,
        )


class CatBoostLearner(Learner):
    """CatBoost with native categoricals; with GLM-boost offsets (``baseline``) it predicts logits.

    CatBoost reports its best iteration as a 0-based index; :attr:`best_iter_base` corrects for that when
    the rounds are chosen inside the fold, and run metadata keeps CatBoost's own number.
    """

    key = "cat"
    booster = True
    best_iter_base = 0
    DEFAULTS: ClassVar[Params] = dict(
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=5,
        iterations=5000,
        eval_metric="AUC",
        od_type="Iter",
        od_wait=200,
    )

    def fit_predict(self, d: FoldData, seed: int = 0, n_iter: int | None = None) -> Prediction:
        """See :meth:`Learner.fit_predict`."""
        from catboost import CatBoostClassifier, Pool

        p = dict(self.DEFAULTS, thread_count=THREADS, verbose=0, random_seed=seed)
        p.update(self.params)
        if n_iter is not None:
            p["iterations"] = n_iter
            p.pop("od_type")
            p.pop("od_wait")
        cats = [c for c in CATS if c in d.X.columns]

        def pool(
            X: pd.DataFrame,
            y: np.ndarray | None = None,
            w: np.ndarray | None = None,
            base: np.ndarray | None = None,
        ) -> Pool:
            X = X.copy()
            for c in cats:
                X[c] = X[c].astype(str)
            return Pool(X, y, weight=w, cat_features=cats, baseline=base)

        raw = "RawFormulaVal"
        m = CatBoostClassifier(**p)
        if n_iter is not None:
            m.fit(pool(d.X, d.y, d.w, None if d.init is None else d.init[0]))
            best = n_iter
        elif d.init is not None:
            m.fit(
                pool(d.X, d.y, d.w, d.init[0]),
                eval_set=pool(d.X_val, d.y_val, base=d.init[1]),
                use_best_model=True,
            )
            best = m.get_best_iteration()
        else:
            m.fit(pool(d.X, d.y, d.w), eval_set=pool(d.X_val, d.y_val), use_best_model=True)
            best = m.get_best_iteration()
        if d.init is not None:
            return Prediction(
                m.predict(pool(d.X_val), prediction_type=raw) + d.init[1],
                m.predict(pool(d.X_test), prediction_type=raw) + d.init[2],
                best,
            )
        return Prediction(m.predict_proba(pool(d.X_val))[:, 1], m.predict_proba(pool(d.X_test))[:, 1], best)


LEARNERS: dict[str, type[Learner]] = {
    cls.key: cls
    for cls in (GLMLearner, MLPLearner, RealMLPLearner, LightGBMLearner, XGBoostLearner, CatBoostLearner)
}


def make_learner(key: str, params: Params | None = None) -> Learner:
    """Instantiate the family named ``key`` (one of ``LEARNERS``).

    Raises:
        ConfigError: unknown family.
    """
    try:
        return LEARNERS[key](params)
    except KeyError:
        raise ConfigError(f"unknown model {key!r}; choose from {sorted(LEARNERS)}") from None

In [ ]:
%%writefile src/train_gbdt.py
r"""Five-fold CV of one model on the shared folds; saves out-of-fold and test predictions to ``runs/<name>/``.

The workhorse of the repository: every own model in docs/DESIGN.md is one command line of this script.

    ../.venv/bin/python train_gbdt.py --model lgb --offset glm --fe art --orig 0 --te single --keys full \
        --te-smooth 5,20,100 --joint glm --key-counts --es inner --seeds 3 --name r11_gb_all_kc

Pipeline per fold (see the README's diagram):

1. feature set (``--fe``, ``features.FEATURE_SETS``) and, with ``--te``, target-encoding keys
   (``features.te_keys``) built once over train, test and original rows, without labels;
2. in-fold target encoding (``encoding.FoldEncoder``): fitted on the fold's training rows only;
3. optional original rows appended with weight ``--orig``; optional GLM-boost offset (``--offset glm``);
4. the model (``--model``, ``learners.LEARNERS``), ``--seeds`` seeds averaged, rounds chosen on the
   validation fold (``--es val``) or inside the training fold (``--es inner``, honest).

Outputs ``runs/<name>/{oof.npy,test.npy,meta.json}`` only for full five-fold runs on all rows; ``--rows``,
``--folds`` below 5 and ``--train-frac`` below 1 are checks that save nothing.
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from common import RUNS, TARGET, ConfigError, folds, load, save_run
from encoding import FoldEncoder, key_counts
from features import FEATURE_SETS, Frames, te_keys
from learners import LEARNERS, FoldData, Learner, Params, Prediction, fit_glm, make_learner


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    """Parse the command line (``argv`` defaults to ``sys.argv[1:]``)."""
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--model", required=True, choices=list(LEARNERS))
    ap.add_argument("--fe", default="raw", choices=list(FEATURE_SETS))
    ap.add_argument("--orig", type=float, default=1.0, help="weight of original rows; 0 drops them")
    ap.add_argument("--name", required=True)
    ap.add_argument(
        "--params",
        default="{}",
        help="JSON overrides of model params; glm_C and glm_splines go to the GLM-boost offset",
    )
    ap.add_argument("--folds", type=int, default=5, help="run only the first k folds (quick checks)")
    ap.add_argument(
        "--te",
        default="none",
        choices=["none", "single", "pairs"],
        help="in-fold target encoding of exact-value keys (single) and also top-10 pairs (pairs)",
    )
    ap.add_argument("--te-smooth", default="auto", help='"auto", m-estimate weight, or list 5,20,100')
    ap.add_argument(
        "--keys",
        default="basic",
        choices=["basic", "full", "tokens"],
        help="full adds the digit-key family; tokens adds GPT-2 tokens and 3-digit chunks on top",
    )
    ap.add_argument(
        "--offset",
        default="none",
        choices=["none", "glm"],
        help="glm: the trees start from an in-fold GLM's logit (GLM boost; lgb, xgb or cat)",
    )
    ap.add_argument(
        "--joint",
        default="none",
        choices=["none", "all", "glm"],
        help="joint keys of non-additive column pairs: for every model (all) or the GLM only (glm)",
    )
    ap.add_argument(
        "--key-counts", action="store_true", help="log count of every key's value over train + test"
    )
    ap.add_argument("--te-cv", type=int, default=5, help="inner cross-fitting folds of the target encoder")
    ap.add_argument(
        "--enc-seed", type=int, default=0, help="seed of the encoders' inner cross-fitting split (D29)"
    )
    ap.add_argument(
        "--hier",
        type=float,
        default=0,
        help="m > 0 adds hierarchical encodings of each float column's key chain (needs --keys full)",
    )
    ap.add_argument(
        "--es",
        default="val",
        choices=["val", "inner"],
        help="boosters: early-stop on the validation fold (val, optimistic OOF; runs before D22) or pick the"
        " rounds on a 10%% holdout of the training fold and refit on all of it (inner, honest OOF)",
    )
    ap.add_argument("--seeds", type=int, default=1, help="models per fold with different seeds, averaged")
    ap.add_argument("--seed0", type=int, default=0, help="first seed; seeds run seed0 .. seed0 + seeds - 1")
    ap.add_argument(
        "--neg-frac",
        type=float,
        default=1.0,
        help="undersampling bag: each seed trains on all frauds and this random share of legitimate rows",
    )
    ap.add_argument("--rows", type=int, default=0, help="smoke test on a stratified subsample; saves nothing")
    ap.add_argument(
        "--train-frac",
        type=float,
        default=1.0,
        help="learning curve: train each fold on this stratified fraction of its training rows, score on the"
        " full validation fold; saves nothing below 1",
    )
    ap.add_argument(
        "--refit-only",
        action="store_true",
        help="skip the CV; fit once on all training rows and write runs/<name>/test_refit.npy next to an"
        " existing run (D34); boosters need --es inner",
    )
    return ap.parse_args(argv)


def check_args(a: argparse.Namespace) -> None:
    """Refuse option combinations that cannot work, before any data is read.

    Raises:
        ConfigError: with the reason.
    """
    learner = LEARNERS[a.model]
    if not 1 <= a.folds <= 5:
        raise ConfigError("--folds must be 1 to 5")
    if a.seeds < 1:
        raise ConfigError("--seeds must be at least 1")
    if not 0 < a.neg_frac <= 1:
        raise ConfigError("--neg-frac must be in (0, 1]")
    if not 0 < a.train_frac <= 1:
        raise ConfigError("--train-frac must be in (0, 1]")
    if a.orig < 0:
        raise ConfigError("--orig must be 0 or more")
    if a.offset == "glm" and (a.orig > 0 or a.te == "none"):
        raise ConfigError("--offset glm needs --orig 0 and target encoding")
    if a.offset == "glm" and not learner.booster:
        raise ConfigError(f"--offset glm needs a booster (lgb, xgb or cat), not {a.model}")
    if a.model == "realmlp" and a.orig > 0:
        raise ConfigError("--model realmlp needs --orig 0 (no sample weights)")
    if a.hier > 0 and (a.te == "none" or a.keys == "basic"):
        raise ConfigError("--hier needs target encoding and --keys full or tokens")
    if a.key_counts and a.te == "none":
        raise ConfigError("--key-counts needs target encoding (--te)")
    if a.refit_only and learner.booster and a.es != "inner":
        raise ConfigError("--refit-only needs --es inner for boosters (no validation fold to stop on)")
    if a.refit_only and (a.rows or a.train_frac < 1 or a.folds < 5):
        raise ConfigError("--refit-only runs on all rows (no --rows, --train-frac or --folds)")
    try:
        params = json.loads(a.params)
    except json.JSONDecodeError as e:
        raise ConfigError(f"--params is not valid JSON: {e}") from None
    if not isinstance(params, dict):
        raise ConfigError("--params must be a JSON object")


@dataclass
class CVResult:
    """Out-of-fold and test predictions of a run, with per-fold scores and rounds."""

    oof: np.ndarray
    test: np.ndarray
    scores: list[float] = field(default_factory=list)
    iters: list[int] = field(default_factory=list)
    seconds: float = 0.0


class Experiment:
    """One ``train_gbdt.py`` run: data, encoder and model, trained fold by fold.

    Args:
        a: parsed and checked command line.
        tr, te, og: frames from ``common.load`` (train possibly subsampled by ``--rows``).
    """

    def __init__(self, a: argparse.Namespace, tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> None:
        self.a = a
        self.y: np.ndarray = tr[TARGET].values
        self.y_og: np.ndarray = og[TARGET].values
        params: Params = json.loads(a.params)
        self.glm_params: Params = {"C": params.pop("glm_C", 0.01), "splines": params.pop("glm_splines", 0)}
        self.learner: Learner = make_learner(a.model, params)
        self.features: Frames = FEATURE_SETS[a.fe](tr, te, og)
        self.encoder: FoldEncoder | None = None
        if a.te != "none":
            keys = te_keys(
                tr,
                te,
                og,
                pairs=a.te == "pairs",
                digit_keys=a.keys in ("full", "tokens"),
                joint=a.joint != "none",
                token_keys=a.keys == "tokens",
            )
            if a.key_counts:
                self.features = key_counts(self.features, keys)
            self.encoder = FoldEncoder(
                keys, self.y, a.te_smooth.split(","), a.te_cv, a.enc_seed, a.joint == "glm", a.hier
            )
            print(f"target-encoding {self.encoder.n_keys} keys in each fold", flush=True)

    def fold_data(self, i_tr: np.ndarray, i_va: np.ndarray, labels_visible: bool) -> FoldData:
        """Training, validation and test frames of one fold, encoded, with original rows and GLM offset."""
        a = self.a
        if self.encoder is None:
            Xtr, Xte, Xog = self.features
            Xa, Xva, Xte_k, Xog_k = Xtr.iloc[i_tr], Xtr.iloc[i_va], Xte, Xog
        else:
            Xa, Xva, Xte_k, Xog_k = self.encoder.transform(self.features, i_tr, i_va)
        ya = self.y[i_tr]
        wa = np.ones(len(i_tr))
        if a.orig > 0:
            Xa = pd.concat([Xa, Xog_k], ignore_index=True)
            ya = np.concatenate([ya, self.y_og])
            wa = np.concatenate([wa, np.full(len(self.y_og), a.orig)])
        init = None
        if a.offset == "glm":
            logits, _ = fit_glm(self.glm_params, Xa, ya, [Xa, Xva, Xte_k])
            init = (logits[0], logits[1], logits[2])
        if not self.learner.on_glm_design:
            # tej_ columns (joint keys for the GLM only) feed the GLM offset but not the trees.
            drop = [c for c in Xa.columns if c.startswith("tej_")]
            Xa, Xva, Xte_k = (X.drop(columns=drop) for X in (Xa, Xva, Xte_k))
        return FoldData(Xa, ya, wa, Xva, self.y[i_va] if labels_visible else None, Xte_k, init)

    def predict_fold(self, k: int, d: FoldData) -> Prediction:
        """Average the fold's predictions over the seeds (each seed with its own undersampling bag)."""
        a, learner = self.a, self.learner
        n_seeds = a.seeds if learner.seeded else 1
        inner = a.es == "inner" and learner.booster
        pv, pt, it = 0.0, 0.0, 0
        for s in range(a.seed0, a.seed0 + n_seeds):
            ds = d
            if a.neg_frac < 1:
                # Undersampling bag (D32): all frauds, a fresh random share of legitimate rows per seed.
                rng = np.random.default_rng(1000 * k + s)
                ds = d.training_subset(np.flatnonzero((d.y == 1) | (rng.random(len(d.y)) < a.neg_frac)))
            n = learner.inner_rounds(ds, seed=s, fold=k) if inner else None
            v, t, i = learner.fit_predict(ds, seed=s, n_iter=n)
            pv, pt, it = pv + v / n_seeds, pt + t / n_seeds, it + int(i) // n_seeds
        return Prediction(np.asarray(pv), np.asarray(pt), int(it))

    def run(self) -> CVResult:
        """Train every requested fold; print each fold's AUC as it finishes."""
        a, y = self.a, self.y
        fl = folds(y)[: a.folds]
        res = CVResult(np.full(len(y), np.nan), np.zeros(len(self.features[1])))
        inner = a.es == "inner" and self.learner.booster
        t0 = time.time()
        for k, (i_tr, i_va) in enumerate(fl):
            if a.train_frac < 1:
                sub = train_test_split(i_tr, train_size=a.train_frac, stratify=y[i_tr], random_state=k)[0]
                i_tr = np.sort(sub)
            p = self.predict_fold(k, self.fold_data(i_tr, i_va, labels_visible=not inner))
            res.oof[i_va] = p.val
            res.test += p.test / len(fl)
            res.scores.append(roc_auc_score(y[i_va], p.val))
            res.iters.append(p.n_iter)
            print(f"fold {k} auc {res.scores[-1]:.5f} iter {p.n_iter} ({time.time() - t0:.0f}s)", flush=True)
        res.seconds = time.time() - t0
        return res

    def refit_test(self) -> Prediction:
        """Test predictions of one model fitted on all training rows (D34).

        The training rows' encodings are cross-fitted as in every fold; the test rows are encoded from all
        training rows. Boosters choose their rounds on a 10% holdout (``--es inner``). The fold machinery
        needs a validation set, so the first training row stands in for one; its prediction is discarded.
        """
        rows = np.arange(len(self.y))
        return self.predict_fold(99, self.fold_data(rows, rows[:1], labels_visible=False))


def main(argv: list[str] | None = None) -> int:
    """Run the CV, print the scores and save the run when it is a full one; return the exit status."""
    a = parse_args(argv)
    try:
        check_args(a)
        tr, te, og = load()
        if a.rows:
            tr = train_test_split(tr, train_size=a.rows, stratify=tr[TARGET], random_state=0)[0]
            tr, te = tr.reset_index(drop=True), te.iloc[:1000].reset_index(drop=True)
        if a.refit_only and not (RUNS / a.name / "oof.npy").exists():
            raise ConfigError(f"--refit-only needs the run {a.name!r} in {RUNS}")
        exp = Experiment(a, tr, te, og)
        if a.refit_only:
            t0 = time.time()
            p = exp.refit_test()
            np.save(RUNS / a.name / "test_refit.npy", p.test)
            info = dict(vars(a), refit_rounds=p.n_iter, seconds=round(time.time() - t0))
            (RUNS / a.name / "refit.json").write_text(json.dumps(info, indent=2))
            print(f"{a.name}: wrote test_refit.npy ({p.n_iter} rounds, {time.time() - t0:.0f}s)")
            return 0
        res = exp.run()
    except (ConfigError, FileNotFoundError) as e:
        print(f"train_gbdt.py: error: {e}", file=sys.stderr)
        return 2
    y = exp.y
    mask = ~np.isnan(res.oof)
    cv = roc_auc_score(y[mask], res.oof[mask])
    print(
        f"{a.name}: CV {cv:.5f}  folds {np.round(res.scores, 5).tolist()}  "
        f"mean {np.mean(res.scores):.5f} sd {np.std(res.scores):.5f}"
    )
    if a.folds == 5 and not a.rows and a.train_frac == 1:
        meta = dict(vars(a), cv=cv, folds=res.scores, iters=res.iters, seconds=round(res.seconds))
        save_run(a.name, res.oof, res.test, meta)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile src/stack.py
r"""Stack saved runs and score every stacker by nested CV; optionally write a submission.

    ../.venv/bin/python stack.py RUN [RUN ...] [--method {logistic,hill}] [--C 0.01] [--no-hill] \
        [--oof-out X.npy] [--out ../submissions/X.csv]

Stackers, each fitted on four of the members' five folds and scored on the fifth, so its CV is honest:

* rank average (fit-free; printed for reference);
* logistic regression over member logits at C 0.01, 0.1, 1 and ``--C``;
* greedy hill climbing: forward selection with replacement on standardised logits, maximising AUC, then
  a one-feature logistic calibration (skip with ``--no-hill``; slow on many members).

Runs saved as logits (GLM, GLM boost) are detected and used as they are (:func:`to_logit`). Runs with NaN
OOF (probes) are scored only on their non-NaN rows and cannot be stacked into a submission. ``--method`` and
``--C`` choose the stacker that writes the submission (fitted on all rows) and ``--oof-out`` (its nested
out-of-fold scores, the input of ``zones.py``, ``stack_average.py`` and ``residual.py``).
"""

from __future__ import annotations

import argparse
import json
import sys
from collections.abc import Callable, Sequence
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from common import RUNS, TARGET, CVScore, Fold, folds, load, load_run, logit

FitScore = Callable[[np.ndarray, np.ndarray], np.ndarray]
"""A stacker: fit on the training-row positions, return scores for the validation-row positions."""


def to_logit(oof: np.ndarray, test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Logit of a run's predictions; runs that already hold logits (GLM, GLM boost) pass through.

    A run is on the logit scale when any OOF value lies outside [0, 1]; OOF and test always share a scale.
    """
    v = oof[~np.isnan(oof)]
    if v.min() < 0 or v.max() > 1:
        return oof, test
    return logit(oof), logit(test)


def member_logits(loaded: Sequence[tuple[np.ndarray, np.ndarray]]) -> tuple[np.ndarray, np.ndarray]:
    """Stack (OOF, test) score pairs as logits: (OOF rows x members, test rows x members)."""
    pairs = [to_logit(o, t) for o, t in loaded]
    return np.column_stack([p[0] for p in pairs]), np.column_stack([p[1] for p in pairs])


def load_members(runs: Sequence[str]) -> tuple[np.ndarray, np.ndarray]:
    """Member logits of saved runs (see :func:`member_logits`)."""
    return member_logits([load_run(r) for r in runs])


def nested(
    fl: Sequence[Fold], y: np.ndarray, fit_score: FitScore, rows: np.ndarray | None = None
) -> tuple[np.ndarray, CVScore]:
    """Out-of-fold predictions of a stacker and their score.

    Args:
        fl: the folds.
        y: labels.
        fit_score: the stacker (see :data:`FitScore`).
        rows: optional mask of usable rows; the others are left NaN and unscored.
    """
    pred = np.full(len(y), np.nan)
    for i_tr, i_va in fl:
        if rows is not None:
            i_tr, i_va = i_tr[rows[i_tr]], i_va[rows[i_va]]
        pred[i_va] = fit_score(i_tr, i_va)
    return pred, CVScore.of(y, pred, fl, rows)


def hill_climb(Z: np.ndarray, y: np.ndarray, steps: int = 40) -> np.ndarray:
    """Greedy forward selection with replacement maximising AUC; returns weights summing to 1.

    Z holds standardised logits (one column per member). Starts from the best single member and stops
    when no addition improves AUC by more than 1e-6, or after ``steps`` additions.
    """
    counts = np.zeros(Z.shape[1])
    aucs = [roc_auc_score(y, Z[:, j]) for j in range(Z.shape[1])]
    counts[int(np.argmax(aucs))] = 1
    cur, best = Z[:, int(np.argmax(aucs))].copy(), max(aucs)
    for _ in range(steps):
        trial = [roc_auc_score(y, cur + Z[:, j]) for j in range(Z.shape[1])]
        j = int(np.argmax(trial))
        if trial[j] <= best + 1e-6:
            break
        cur, best = cur + Z[:, j], trial[j]
        counts[j] += 1
    return counts / counts.sum()


def calibrate(score: np.ndarray, y: np.ndarray) -> LogisticRegression:
    """One-feature logistic fit, so blends fitted on different folds share one scale when pooled."""
    return LogisticRegression(C=1e6, max_iter=1000).fit(score[:, None], y)


def standardise(Z: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Scale each column of Z by the mean and SD of the same column in ref."""
    return (Z - ref.mean(axis=0)) / ref.std(axis=0)


def logistic_stacker(Z: np.ndarray, y: np.ndarray, C: float) -> FitScore:
    """L2 logistic regression on member logits at regularisation ``C``."""

    def fit_score(i_tr: np.ndarray, i_va: np.ndarray) -> np.ndarray:
        return LogisticRegression(C=C, max_iter=1000).fit(Z[i_tr], y[i_tr]).decision_function(Z[i_va])

    return fit_score


def hill_stacker(Z: np.ndarray, y: np.ndarray) -> FitScore:
    """Hill climbing on standardised logits, calibrated on the training rows (:func:`hill_climb`)."""

    def fit_score(i_tr: np.ndarray, i_va: np.ndarray) -> np.ndarray:
        w = hill_climb(standardise(Z[i_tr], Z[i_tr]), y[i_tr])
        cal = calibrate(standardise(Z[i_tr], Z[i_tr]) @ w, y[i_tr])
        return cal.decision_function((standardise(Z[i_va], Z[i_tr]) @ w)[:, None])

    return fit_score


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    """Parse the command line (``argv`` defaults to ``sys.argv[1:]``)."""
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("runs", nargs="+", help="run names under runs/")
    ap.add_argument("--out", type=Path, help="submission CSV to write (plus a JSON with the spec)")
    ap.add_argument("--method", default="logistic", choices=["logistic", "hill"])
    ap.add_argument("--C", type=float, default=1.0, help="logistic stacker regularisation")
    ap.add_argument("--no-hill", action="store_true", help="skip hill climbing (slow on many members)")
    ap.add_argument("--oof-out", type=Path, help="save nested OOF stack scores (.npy)")
    ap.add_argument(
        "--refit-test",
        action="store_true",
        help="test scores from test_refit.npy where a run has one (D34); the OOF scores are unchanged",
    )
    a = ap.parse_args(argv)
    if a.method == "hill" and a.no_hill:
        ap.error("--method hill needs hill climbing (drop --no-hill)")
    return a


def main(argv: list[str] | None = None) -> int:
    """Score every stacker by nested CV and write the chosen one's outputs; return the exit status."""
    a = parse_args(argv)
    tr, te, _ = load()
    y = tr[TARGET].to_numpy()
    try:
        loaded = [load_run(r, refit=a.refit_test) for r in a.runs]
    except FileNotFoundError as e:
        print(f"stack.py: error: {e}", file=sys.stderr)
        return 2
    oof = np.column_stack([o for o, _ in loaded])
    Z, Zt = member_logits(loaded)
    m = ~np.isnan(oof).any(axis=1)
    print(f"rows scored: {m.sum()} of {len(y)}")
    if a.refit_test:
        refit = [r for r in a.runs if (RUNS / r / "test_refit.npy").exists()]
        print(f"test scores from one refit on all rows for {len(refit)} of {len(a.runs)} members")
    for i, r in enumerate(a.runs):
        print(f"  {r:28s} {roc_auc_score(y[m], oof[m, i]):.5f}")

    rank = np.column_stack([rankdata(oof[m, i]) for i in range(oof.shape[1])]).mean(axis=1)
    print(f"  {'rank average':28s} {roc_auc_score(y[m], rank):.5f}")

    fl = folds(y)
    stackers: dict[str, FitScore] = {
        f"logistic C={C:g}": logistic_stacker(Z, y, C) for C in sorted({0.01, 0.1, 1.0, a.C})
    }
    if not a.no_hill:
        stackers["hill climbing"] = hill_stacker(Z, y)
    preds: dict[str, np.ndarray] = {}
    results: dict[str, CVScore] = {}
    for key, fit_score in stackers.items():
        preds[key], results[key] = nested(fl, y, fit_score, m)
        print(f"  {key + ' (nested)':28s} {results[key]}")

    lr = LogisticRegression(C=a.C, max_iter=1000).fit(Z[m], y[m])
    print("  logistic weights", dict(zip(a.runs, np.round(lr.coef_[0], 3).tolist(), strict=True)))
    if a.method == "hill":
        key = "hill climbing"
        w = hill_climb(standardise(Z[m], Z[m]), y[m])
        print("  hill weights", dict(zip(a.runs, np.round(w, 3).tolist(), strict=True)))
        cal = calibrate(standardise(Z[m], Z[m]) @ w, y[m])
        final = cal.decision_function((standardise(Zt, Z[m]) @ w)[:, None])
    else:
        key = f"logistic C={a.C:g}"
        final = lr.decision_function(Zt)

    if a.oof_out:
        np.save(a.oof_out, preds[key])
        print(f"wrote {a.oof_out}")
    if a.out:
        if not m.all() or np.isnan(Zt).any():
            print(
                "stack.py: error: cannot write a submission from probe runs with missing predictions",
                file=sys.stderr,
            )
            return 2
        sub = pd.DataFrame({"id": te["id"], TARGET: 1 / (1 + np.exp(-final))})
        a.out.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(a.out, index=False)
        spec = {
            "runs": a.runs,
            "method": a.method,
            "C": a.C,
            "cv": results[key].cv,
            "folds": list(results[key].folds),
        }
        a.out.with_suffix(".json").write_text(json.dumps(spec, indent=2))
        print(f"wrote {a.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile src/zones.py
"""Zero-fraud zones (D27): test them on a stack's nested out-of-fold scores and apply them to a submission.

A zone is a set of rows with no fraud in the training data; sending its test rows to the bottom of the
ranking can only help AUC if the zone also holds no fraud on unseen rows. The zones (public notebooks of
daoviet, goodpjw2008 and Weni Mohamed, 2026-11-18/19) were found on all training rows, so scoring them on
the same rows is optimistic. Here every merchant-risk edge is refitted on the four training folds (just
below the lowest fraud inside the zone's other conditions) and applied to the held-out fold:

  zone 1:  merchant_risk_score < 13.1
  zone 2:  velocity_score <= 4.1 and merchant_risk_score <= 20
  zone 34: all eight risk flags clear and (velocity_score <= 4.1 and merchant_risk_score <= 33.8,
           or velocity_score <= 6.0 and merchant_risk_score <= 25)

The velocity edges stay as published (chosen on all rows, a small optimism this test does not remove).
For the test set each edge is the lower of the published one and the one refitted on all training rows.

Other scripts reuse :func:`hand_zones` (zone rows of a held-out fold) and :func:`send_to_bottom`.

Usage (from src/):
  ../.venv/bin/python stack.py --no-hill --C 0.1 --oof-out X.npy --out X.csv <runs...>
  ../.venv/bin/python zones.py --oof X.npy --sub X.csv [--out X_zones.csv]
"""

from __future__ import annotations

import argparse
import sys
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from common import TARGET, CVScore, Fold, folds, load

FLAGS_ZERO = ["cvv_retry_count", "prior_disputes"]
FLAGS_OFF = [
    "is_new_merchant",
    "is_foreign_transaction",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]


def clean(d: pd.DataFrame) -> np.ndarray:
    """Rows with all eight risk flags clear."""
    ok = np.ones(len(d), bool)
    for c in FLAGS_ZERO:
        ok &= d[c].to_numpy() == 0
    for c in FLAGS_OFF:
        ok &= d[c].to_numpy().astype(int) == 0
    return ok


@dataclass(frozen=True)
class Zone:
    """A base condition plus an upper edge on merchant_risk_score.

    Attributes:
        name: printed name.
        base: rows satisfying the conditions other than the merchant-risk edge.
        published: the edge as published.
        strict: the published edge is ``<`` rather than ``<=``.
    """

    name: str
    base: Callable[[pd.DataFrame], np.ndarray]
    published: float
    strict: bool

    def refit(self, d: pd.DataFrame, y: np.ndarray) -> tuple[float, bool]:
        """Edge just below the lowest fraud inside the base condition, capped at the published edge.

        Returns the edge and whether it is strict.
        """
        m = d["merchant_risk_score"].to_numpy()
        fraud = self.base(d) & (y == 1)
        lowest = m[fraud].min() if fraud.any() else np.inf
        return (lowest, True) if lowest <= self.published else (self.published, self.strict)

    def contains(self, d: pd.DataFrame, edge: float, strict: bool) -> np.ndarray:
        """Rows inside the zone for a given merchant-risk edge."""
        m = d["merchant_risk_score"].to_numpy()
        return self.base(d) & ((m < edge) if strict else (m <= edge))


ZONES: tuple[Zone, ...] = (
    Zone("zone 1", lambda d: np.ones(len(d), bool), 13.1, True),
    Zone("zone 2", lambda d: d["velocity_score"].to_numpy() <= 4.1, 20.0, False),
    Zone("zone 3", lambda d: clean(d) & (d["velocity_score"].to_numpy() <= 4.1), 33.8, False),
    Zone("zone 4", lambda d: clean(d) & (d["velocity_score"].to_numpy() <= 6.0), 25.0, False),
)


def hand_zones(
    tr: pd.DataFrame, y: np.ndarray, i_tr: np.ndarray, i_va: np.ndarray, zones: Sequence[Zone] = ZONES
) -> np.ndarray:
    """Mask over the held-out rows ``i_va`` of those inside any zone, edges refitted on rows ``i_tr``."""
    d_tr, d_va = tr.iloc[i_tr], tr.iloc[i_va]
    z = np.zeros(len(i_va), bool)
    for zone in zones:
        z |= zone.contains(d_va, *zone.refit(d_tr, y[i_tr]))
    return z


def send_to_bottom(pred: np.ndarray, fl: Sequence[Fold], masks: Sequence[np.ndarray]) -> np.ndarray:
    """Copy of out-of-fold scores with each fold's masked validation rows set below every score."""
    out = pred.copy()
    low = pred.min() - 1.0
    for (_, i_va), z in zip(fl, masks, strict=True):
        out[i_va[z]] = low
    return out


def zones_on_test(tr: pd.DataFrame, y: np.ndarray, te: pd.DataFrame) -> np.ndarray:
    """Mask of test rows inside any zone, each edge the lower of the published and the refitted one."""
    z = np.zeros(len(te), bool)
    for zone in ZONES:
        edge, s = zone.refit(tr, y)
        zt = zone.contains(te, edge, s)
        print(f"  test {zone.name}: merchant_risk_score {'<' if s else '<='} {edge:g}, {zt.sum()} rows")
        z |= zt
    print(f"  test rows in any zone: {z.sum()} of {len(te)}")
    return z


def main(argv: list[str] | None = None) -> int:
    """Score zones 1, 1+2, 1+2+3 and all four on the OOF scores; apply all four to the submission."""
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--oof", type=Path, required=True, help="nested OOF stack scores from stack.py --oof-out")
    ap.add_argument("--sub", type=Path, help="submission CSV of the same stack")
    ap.add_argument("--out", type=Path, help="write the submission with all zones sent to the bottom")
    a = ap.parse_args(argv)

    tr, te, _ = load()
    y = tr[TARGET].to_numpy()
    pred = np.load(a.oof, allow_pickle=False)
    if pred.shape != (len(tr),) or not np.isfinite(pred).all():
        print(f"zones.py: error: {a.oof} must hold {len(tr)} finite scores", file=sys.stderr)
        return 2
    fl = folds(y)
    base = CVScore.of(y, pred, fl)
    print(f"  stack                     {base}")

    for upto in range(1, len(ZONES) + 1):
        use = ZONES[:upto]
        masks = [hand_zones(tr, y, i_tr, i_va, use) for i_tr, i_va in fl]
        rows = sum(int(z.sum()) for z in masks)
        frauds = sum(int(y[i_va][z].sum()) for (_, i_va), z in zip(fl, masks, strict=True))
        score = CVScore.of(y, send_to_bottom(pred, fl, masks), fl)
        names = "+".join(zone.name.split()[1] for zone in use)
        print(
            f"  + zones {names:17s} {score}"
            f"  up {score.up(base)}/5, {rows} held-out rows in zones, {frauds} frauds among them"
        )

    if a.sub is not None:
        sub = pd.read_csv(a.sub)
        if not np.array_equal(sub["id"].to_numpy(), te["id"].to_numpy()):
            print(f"zones.py: error: {a.sub} ids differ from test.csv", file=sys.stderr)
            return 2
        z = zones_on_test(tr, y, te)
        if a.out is not None:
            p = sub[TARGET].to_numpy().copy()
            p[z] *= 1e-6
            sub[TARGET] = p
            sub.to_csv(a.out, index=False)
            print(f"wrote {a.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
sys.path.insert(0, "src")
from common import RUNS, TARGET, CVScore, folds, load

tr, te, og = load()
y = tr[TARGET].to_numpy()
FOLDS = folds(y)
print(f"train {tr.shape}, test {te.shape}, original {og.shape}")

<a id="s2"></a>

## 2 · Data and metric

700k synthetic training rows, generated from the 20,000-row
[Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026)
dataset (CC0). Fraud is rare, so ROC-AUC is decided by how the 1.7% of frauds rank against the rest.

In [ ]:
print(f"fraud rate: train {y.mean():.4f} ({y.sum():,} frauds), original {og[TARGET].mean():.4f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, c in zip(ax, ["merchant_risk_score", "velocity_score"]):
    r = tr.groupby(pd.qcut(tr[c], 10, duplicates="drop"), observed=True)[TARGET].mean()
    a.bar(range(len(r)), 100 * r.to_numpy(), color="#3b6ea8")
    a.set_xticks(range(len(r)), [f"{iv.right:.0f}" for iv in r.index], fontsize=8)
    a.set_title(f"fraud rate (%) by decile of {c}"); a.set_xlabel("decile upper edge")
plt.tight_layout(); plt.show()
flags = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch",
         "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
print(pd.DataFrame({f: tr.groupby(f)[TARGET].mean() for f in flags}).T.rename(columns={0: "flag 0", 1: "flag 1"}).round(4))

Fraud rises smoothly with both scores and with every flag. No single column is decisive, so the work is
in how each column's values are encoded. Two facts about the synthetic data shape the features below:

- **Exact values repeat.** The generator copies values from the original rows, so a value's own fraud
  rate (a target encoding of the exact value, its integer part, its decimals) carries signal.
- **Pooled artefacts.** Whether a value occurs in the original data at all, and how often it occurs in
  train + test, also carries signal; per-value encodings cannot see it, because each rare value is
  smoothed towards the base rate on its own. These are unsupervised features (`--fe art`).

<a id="s3"></a>

## 3 · Folds, and the one rule

`StratifiedKFold(5, shuffle=True, random_state=42)` on `is_fraud`, the split most public out-of-fold
files also use. **Everything learned from labels is learned inside the training fold**:

- target encodings are cross-fitted inside the training fold (5 inner folds), never on the scored fold;
- boosters pick their number of rounds on a 10% holdout of the training fold, then refit on all of it
  (`--es inner`). Early-stopping on the scored fold, which many notebooks do, inflated our boosters' CV
  by 0.00004 to 0.00009 each (measured in the repo);
- the stacker in section 7 is itself cross-validated (nested).

`run()` trains one model with `train_gbdt.py` and prints its fold scores.

In [ ]:
RESULTS = {}

def run(name, args):
    # train one model unless its predictions already exist, then report its fold AUCs
    d = RUNS / name
    if not (d / "oof.npy").exists():
        t0 = time.time()
        cmd = [sys.executable, "train_gbdt.py", "--name", name, *args.split()]
        p = subprocess.run(cmd, cwd="src", capture_output=True, text=True)
        if p.returncode:
            print(p.stdout[-3000:], p.stderr[-3000:]); raise RuntimeError(name)
        print(f"{name}: trained in {(time.time() - t0) / 60:.0f} min")
    m = json.loads((d / "meta.json").read_text())
    RESULTS[name] = dict(cv=m["cv"], folds=m["folds"])
    print(f"{name}: CV {m['cv']:.5f}  folds {np.round(m['folds'], 5).tolist()}")

<a id="s4"></a>

## 4 · Three GLMs

A logistic regression is a strong model here once each value is encoded well. The design: cubic splines
(8 knots) on every numeric column, one-hot categories, and in-fold target encodings of many **keys**:
each column's exact value, its integer part, its value rounded to tens, its decimal ending, and (with
`--keys full`) its leading digits, its value at 0.1 resolution and the length of its printed value.

**4a. Raw columns and the value keys** (`--fe raw --keys full`, one smoothing weight).

In [ ]:
run("glm_keys", '--model glm --fe raw --te single --keys full --te-smooth 20 --orig 0 --params {"splines":8}')

**4b. Add the pooled artefacts, three smoothing weights and joint keys.** Each key is encoded with
m-estimate weights 5, 20 and 100 (rare values trusted less or more), and eight column pairs whose effect
is not additive become joint keys (the pairs measured and published in megayak's notebook
*S6E11 | Per-Value + Digit TE | Shallow-Tree Stack*, Apache-2.0).

In [ ]:
run("glm_art", '--model glm --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --params {"splines":8}')

**4c. Add GPT-2 token keys.** The language-model notebooks at the top of this competition read each row
as text. Part of what they see is how GPT-2's tokeniser splits a number: `1371` becomes `" 13"` and
`"71"`. Here the first two tokens of each float column's integer part, and its fixed 3-digit chunks,
become keys for the same target encoding (idea after aphoristicartist's and busyaprime's notebooks;
written independently). The GLM gets part of the language models' view at a fraction of the cost.

In [ ]:
run("glm_tok", '--model glm --fe art --te single --keys tokens --te-smooth 5,20,100 --orig 0 --joint glm --params {"splines":8}')

<a id="s5"></a>

## 5 · Two shallow boosters

Deep trees overfit the encodings; 4-leaf trees with a low learning rate do best. Both boosters see the
same encoded keys plus the log count of every key's value over train + test (`--key-counts`).

**5a. LightGBM, 4 leaves.**

In [ ]:
LGB = '{"num_leaves":4,"learning_rate":0.03,"feature_fraction":0.5,"min_child_samples":100,"lambda_l2":1.0,"n_estimators":20000}'
run("lgb_l4", f"--model lgb --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --es inner --key-counts --params {LGB}")

**5b. GLM boost.** The same trees, but started from an in-fold GLM's logit (`--offset glm`): the GLM
carries the smooth part, the trees only learn the interactions it misses. This is our best single model.

In [ ]:
run("glm_boost", f"--model lgb --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --es inner --key-counts --offset glm --params {LGB}")

<a id="s6"></a>

## 6 · RealMLP on value-bucket embeddings

A different view for the stack: no target encodings at all. Every numeric column's integer part and its
two decimals become categories, and RealMLP (pytabkit) learns an embedding per value bucket
(`--fe emb`). Hyperparameters start from yekenot's tuned RealMLP notebook, with 8 ensemble members
instead of 16 for CPU, 3 epochs and no early stopping (the scored fold never picks the model).

In [ ]:
RMLP = ('{"n_ens":8,"n_epochs":3,"batch_size":288,"use_early_stopping":false,"val_metric_name":"1-auc_ovr",'
        '"lr":0.073,"wd":0.038,"sq_mom":0.988,"lr_sched":"flat_anneal","wd_sched":"cos_log_15",'
        '"first_layer_lr_factor":0.25,"embedding_size":5,"max_one_hot_cat_size":16,"hidden_sizes":[512,256,128],'
        '"act":"silu","p_drop":0.05,"p_drop_sched":"pow5","plr_hidden_1":16,"plr_hidden_2":8,"plr_act_name":"gelu",'
        '"plr_lr_factor":0.1151,"plr_sigma":0.75,"ls_eps":0.01,"ls_eps_sched":"sqrt_cos","add_front_scale":false,'
        '"bias_init_mode":"neg-uniform-dynamic-2","tfms":["one_hot","median_center","robust_scale","smooth_clip",'
        '"embedding","l2_normalize"]}')
run("realmlp_emb", f"--model realmlp --fe emb --te none --orig 0 --params {RMLP}")

In [ ]:
table = pd.DataFrame({k: [v["cv"], *v["folds"]] for k, v in RESULTS.items()},
                     index=["CV", "fold 0", "fold 1", "fold 2", "fold 3", "fold 4"]).T
table.round(5)

<a id="s7"></a>

## 7 · The stack

A logistic regression on the six models' logits. It is **nested**: for each outer fold it is fitted on
the other four folds' out-of-fold predictions and scored on the fifth, so its CV is as honest as the
members'. Regularisation C = 0.1.

In [ ]:
from stack import load_members, logistic_stacker, nested

names = list(RESULTS)
Z, Zt = load_members(names)  # logits; the GLMs and the GLM boost already save logits
stack_oof, stack_score = nested(FOLDS, y, logistic_stacker(Z, y, C=0.1))
best = max(RESULTS, key=lambda k: RESULTS[k]["cv"])
print(f"best single ({best}): {RESULTS[best]['cv']:.5f}")
print(f"nested stack:        {stack_score}")
lr = LogisticRegression(C=0.1, max_iter=1000).fit(Z, y)
stack_test = lr.decision_function(Zt)
print("weights:", dict(zip(names, np.round(lr.coef_[0], 3))))

**How noisy is the public leaderboard?** Draw 60,000 rows (the size of the public part of the test set)
from the stack's out-of-fold predictions many times and look at how much the AUC moves.

In [ ]:
rng = np.random.default_rng(0)
single = np.load(RUNS / best / "oof.npy")
level, diff = [], []
for _ in range(300):
    s = rng.choice(len(y), 60_000, replace=False)
    a = roc_auc_score(y[s], stack_oof[s]); level.append(a)
    diff.append(a - roc_auc_score(y[s], single[s]))
full_gain = stack_score.cv - RESULTS[best]["cv"]
print(f"stack AUC on 60,000 random rows: standard deviation {np.std(level):.5f}")
print(f"stack minus best single, all 700k rows: {full_gain:+.5f}")
print(f"stack minus best single, 60,000 rows:   mean {np.mean(diff):+.5f}, standard deviation {np.std(diff):.5f}, "
      f"below zero in {np.mean(np.array(diff) < 0):.0%} of draws")

Two kinds of noise. The **level** of one file's score moves by the first standard deviation between the
public 20% and the private 80%, so a public score says little about the private one at the third
decimal. The **difference** between two files scored on the same rows moves much less, because both see
the same frauds; but even a real, all-folds gain like the stack's over its best member is sometimes
reversed on 60,000 rows. Steps of 0.00001 to 0.0001, as at the top of the board, are well inside that
noise. Five folds of 700,000 rows are not.

**What a blend fitted to the public board does on the private one.** In late November the top of the
public board (0.957 and up) is built by *board models*: blends of a few hundred scored submissions whose
weights are chosen to raise the public score, then pushed further along the same direction. No
out-of-fold check exists for them. The repo rebuilds that situation on the training rows, where the labels
are known (`src/research_board_sim.py`, D35): a random 60,000 rows play the public part, the other 640,000
the private part, and 271 out-of-fold members play the scored files. A logistic blend fitted on the
60,000 "public" rows only, against the honest four-stack mean, over 20 random splits:

| Blend fitted on the public rows | Gain on the public rows | Gain on the private rows | Private up |
| --- | --- | --- | --- |
| mild fit (C = 0.01) | +0.00036 | -0.00048 | 0 of 20 |
| medium fit (C = 0.1) | +0.00197 | -0.00074 | 0 of 20 |
| medium fit, pushed 45% further | +0.00226 | -0.00165 | 0 of 20 |
| strong fit (C = 1) | +0.00361 | -0.00204 | 0 of 20 |

Fitting on the public rows' labels is more than a board model can learn from a few hundred scores, so
the sizes here are an upper bound; the sign is the finding. Every public gain came with a private loss,
and pushing further made the loss larger. The same honest file's public score also moved by 0.0029 (one
standard deviation) from split to split, which is wider than the whole top of the board.

<a id="s8"></a>

## 8 · Zero-fraud zones

Public notebooks (daoviet, goodpjw2008, Weni Mohamed) found four regions with no fraud among the
training rows:

- zone 1: `merchant_risk_score < 13.1`
- zone 2: `velocity_score <= 4.1` and `merchant_risk_score <= 20`
- zone 3: all eight risk flags clear (`cvv_retry_count` and `prior_disputes` 0, the six binary flags 0),
  `velocity_score <= 4.1` and `merchant_risk_score <= 33.8`
- zone 4: all eight flags clear, `velocity_score <= 6.0` and `merchant_risk_score <= 25`

Sending these rows to the bottom of the ranking helps only if they hold no fraud on unseen rows too.
They were found on all training rows, so scoring them on the same rows would flatter them. Here each
merchant-risk edge is refitted on the four training folds (just below the lowest fraud inside the zone,
never above the published edge) and applied to the held-out fold. The velocity edges stay as published,
a small optimism this check does not remove.

In [ ]:
from zones import hand_zones, send_to_bottom, zones_on_test

masks = [hand_zones(tr, y, i_tr, i_va) for i_tr, i_va in FOLDS]  # edges refitted on the training folds
zoned_score = CVScore.of(y, send_to_bottom(stack_oof, FOLDS, masks), FOLDS)
n_fraud = sum(int(y[i_va][z].sum()) for (_, i_va), z in zip(FOLDS, masks))
print(f"held-out rows in a zone: {sum(int(z.sum()) for z in masks):,}, frauds among them: {n_fraud}")
print(f"stack:         {stack_score}")
print(f"stack + zones: {zoned_score}")
print(f"folds up: {zoned_score.up(stack_score)} of 5")

A small gain, but on every fold, and no held-out fraud falls into a refitted zone. The stack already
ranks these rows low; the zones only finish the job.

<a id="s9"></a>

## 9 · What did not help

Measured in the repo, mostly on our full 24-model own stack (nested CV 0.95138), with the rule that a
change is kept only if the stack improves on at least 4 of the 5 folds. Details and every other run:
[docs/DESIGN.md](https://github.com/kabartay/playground-series-s6e11/blob/main/docs/DESIGN.md).

| Idea | Result |
| --- | --- |
| Our own target-first GPT-2 (rows as text), 1 epoch | 0.94396 alone; stack weight about 0, stack 0.95134 |
| The same, 3 epochs on 5 GPUs | 0.94717 alone (+0.003 on every fold); stack 0.95131, 2 of 5 folds up |
| Seed bags of the RealMLPs (3 seeds) | each member +0.0001; stack 0.95139, 1 of 5 folds up |
| RealMLPs trained on all rows instead of a 90% split | members +0.0004; stack 0.95137 |
| A residual LightGBM on raw features, started from the stack | stops after 1 to 7 rounds; 0.95137 |
| The 20,000 original rows added to training | LightGBM -0.0001 (0.94680 against 0.94690) |
| Boosters early-stopped on the scored fold | CV inflated by 0.00004 to 0.00009 per booster |
| FinTFM 1.0.0 linear readout (a pretrained tabular transformer), out-of-fold on the same folds | 0.94054 alone; stack 0.95135, 3 of 5 folds up, small negative weight |
| Smarter stackers: pairwise products, one stacker per segment, rank-Gaussian or disagreement features | 0.95091 to 0.95127, all below the plain logistic stack |
| Zero-fraud zones found by a shallow tree instead of the hand rules of section 8 | 0.93214 to 0.95015, 0 of 5 folds up |
| Three target-encoder seeds for the token GLM, averaged | stack 0.95140, 4 of 5 folds up but only +0.00002 |
| The GLMs solved to a 1,000 times tighter tolerance | the same fold AUC (0.94957) as the default |
| XGBoost or CatBoost GLM boost added to this notebook's six | stack + zones 0.95130 to 0.95137 against 0.95135, at most 3 of 5 folds up |
| Test predictions from one refit on all rows instead of the mean of the five fold models | simulated one level down: GLM -0.00010 (1 of 5 folds), GLM boost +0.00010 but 3 of 5 and following its round count |
| Undersampling bags (every fraud, a fresh 25% of the legitimate rows per seed) | 0.95033 against 0.95043 for the plain booster |
| Mean and SD of other columns per exact value (IEEE-CIS-style "client" aggregates) | -0.0008, 0 of 5 folds |
| Isolation Forest and PCA reconstruction error as features or members | 0.811 alone without labels; nothing added once 11,689 frauds are labelled |

The pattern: once the stack has a GLM, a GLM boost and a network, a new member has to see something none of
them sees. More of the same, or a stronger version of the same, does not move it.

<a id="s10"></a>

## 10 · Submission

The stack's test logit, with the zone rows (edges refitted on all training rows) sent to the bottom.

In [ ]:
z_test = zones_on_test(tr, y, te)  # each edge: the lower of the published and the refitted-on-all-rows one
p = 1 / (1 + np.exp(-stack_test))
p[z_test] *= 1e-6
sub = pd.DataFrame({"id": te["id"], TARGET: p})
sub.to_csv("submission.csv", index=False)
print(f"test rows in a zone: {z_test.sum():,} of {len(te):,}")
sub.head()

## Credits

- Data: the competition, and [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) (CC0).
- Joint pairs (section 4b): megayak, *S6E11 | Per-Value + Digit TE | Shallow-Tree Stack* (Apache-2.0).
- GPT-2 token keys (4c): the idea after aphoristicartist's and busyaprime's notebooks.
- RealMLP settings and value-bucket embeddings (6): yekenot's RealMLP notebook.
- Zero-fraud zones (8): daoviet, goodpjw2008, Weni Mohamed.

All code here is ours, from [github.com/kabartay/playground-series-s6e11](https://github.com/kabartay/playground-series-s6e11) (MIT), which also logs every run, including the
ones that failed. If something here helps you, an upvote is appreciated.